Install Required Libraries

In [ ]:
!pip install -qU \
langchain==0.3.13 \
langchain-community==0.3.13 \
langchain-core==0.3.63 \
langchain-groq==0.2.3 \
langchain-text-splitters==0.3.4 \
faiss-cpu \
sentence-transformers \
pypdf \
python-dotenv


**Important:** After running the install cell above, go to **Runtime > Restart session** in Colab, then run the notebook from the top. This makes sure the pinned package versions are actually loaded (Colab pre-loads some packages before your `pip install` runs).

Import Libraries

In [ ]:
import os

from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

from langchain_groq import ChatGroq


Upload Course PDF

In [56]:
uploaded = files.upload()

Saving 1_Introduction to CTS.pdf to 1_Introduction to CTS (4).pdf


Load PDF

In [57]:
pdf_file = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_file)

documents = loader.load()

print(f"Pages Loaded: {len(documents)}")

Pages Loaded: 70


Split Text

In [58]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

docs = text_splitter.split_documents(documents)

print(f"Chunks Created: {len(docs)}")

Chunks Created: 70


Embeddings

In [59]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Create Vector Database

In [60]:
vector_db = FAISS.from_documents(
    docs,
    embeddings
)

print("FAISS Database Created Successfully!")

FAISS Database Created Successfully!


Enter OpenAI API Key

In [ ]:
import getpass

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")


Load LLM

In [62]:
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

Create Memory

In [63]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

Create RAG Chain

In [64]:
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vector_db.as_retriever(),
    memory=memory,
    return_source_documents=True
)

Ask Questions

In [65]:
while True:

    question = input("You: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    response = qa_chain.invoke(
        {"question": question}
    )

    print("\nAssistant:")
    print(response["answer"])
    print("-"*60)

You: What is this course about?

Assistant:
This course, EECS203, appears to be about Signals Analysis, which is a subject in the field of Electronics and Communications. The specific topics covered include time domain and frequency domain analysis, as well as the concept of Z transform and its properties.
------------------------------------------------------------
You: Summarize the uploaded PDF.

Assistant:
The uploaded PDF appears to be a collection of slides from a course taught by Prof. Dr. Mohamed Fathy Abu El-Yazeed. The content includes:

1. A grading breakdown for the course (Participation, Quizzes, Assignments, Mid-Semester, and Final Exam).
2. A discussion on the elements of a communication system.
3. A mention of independent variables, including spatial variables in images.
4. Examples of signals.
5. A summary and conclusions section.

It seems to be a course on communication systems or a related topic, but without more context, it's difficult to provide a more detailed su